# SMART-MUSTAHIK: Sistem Pendukung Keputusan Penyaluran Zakat Presisi
## Pemrosesan Data Sosio-Ekonomi BPS Jawa Timur (2020–2024) & Geospasial Satelit NTL

Notebook ini memproses dataset resmi Badan Pusat Statistik (BPS) Provinsi Jawa Timur (38 Kabupaten/Kota, 2020–2024), melakukan imputasi data spasial-temporal, memformulasikan kriteria kelaikan *Mustahik* berbasis **Fiqih Kifayah (QS. At-Taubah: 60)**, serta melatih model Machine Learning Random Forest untuk meminimalkan *Inclusion Error* dan *Exclusion Error*.

In [ ]:
# Import pustaka manipulasi data & pemodelan
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import KNNImputer
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

SEED = 42
np.random.seed(SEED)

os.makedirs('data', exist_ok=True)
os.makedirs('images', exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'


In [ ]:
# Memuat & Membersihkan Dataset BPS Jawa Timur (2020-2024)
csv_path = os.path.join('data', 'dataset_jatim_2020_2024.csv')
df_raw = pd.read_csv(csv_path)

num_cols = [
    'total_population', 'number_of_poor_people', 'poverty_percentage',
    'school_participation_rate', 'sanitation_access_percent',
    'drinking_water_access_percent', 'proper_housing_percent',
    'unemployment_rate', 'gdp_per_capita', 'population_density', 'hdi'
]

df = df_raw.copy()
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
df = df.sort_values(['regency_city', 'year']).reset_index(drop=True)

# Imputasi nilai kosong berbasis interpolasi linier per wilayah
impute_cols = ['gdp_per_capita', 'hdi', 'unemployment_rate', 'poverty_percentage', 'sanitation_access_percent', 'drinking_water_access_percent', 'population_density', 'total_population']
df_imputed = df.copy()
for reg in df_imputed['regency_city'].unique():
    mask = df_imputed['regency_city'] == reg
    df_imputed.loc[mask, impute_cols] = df_imputed.loc[mask, impute_cols].interpolate(method='linear', limit_direction='both')

imputer = KNNImputer(n_neighbors=3)
df_imputed[impute_cols] = imputer.fit_transform(df_imputed[impute_cols])

# Simulasi Radiasi Satelit NTL (VIIRS) terikat PDRB & Kepadatan Penduduk
log_gdp = np.log(df_imputed['gdp_per_capita'])
log_dens = np.log(df_imputed['population_density'])
ntl_base = 2.5 * log_gdp + 1.2 * log_dens - 18.0
df_imputed['radiasi_satelit_ntl'] = np.clip(ntl_base + np.random.normal(0, 1.2, len(df_imputed)), 0.1, 65.0)

print(f"Dataset BPS Terproses: {len(df_imputed)} Observasi (38 Kab/Kota x 5 Tahun)")
display(df_imputed.head())


In [ ]:
# Formulasi Ground Truth Mustahik (QS. At-Taubah: 60)
HAD_KIFAYAH = 1950000  # Had Kifayah bulanan per kapita (Rp)

# Estimasi pendapatan bulanan & skor aset
df_imputed['pendapatan_per_kapita_bulan'] = (df_imputed['gdp_per_capita'] * 1000) / 12.0
df_imputed['skor_aset'] = np.clip(
    0.4 * df_imputed['hdi'] + 0.3 * df_imputed['sanitation_access_percent'] + 0.3 * (df_imputed['radiasi_satelit_ntl'] / 65.0 * 100),
    10, 98
)

# Kriteria Mustahik: Tingkat kemiskinan > 9.5% ATAU Pendapatan < Had Kifayah
df_imputed['status_mustahik'] = np.where(
    (df_imputed['poverty_percentage'] > 9.5) | (df_imputed['pendapatan_per_kapita_bulan'] < HAD_KIFAYAH),
    1, 0
)

dist = df_imputed['status_mustahik'].value_counts()
print(f"Mustahik (Fakir/Miskin) : {dist.get(1, 0)} Observasi ({dist.get(1, 0)/len(df_imputed)*100:.1f}%)")
print(f"Non-Mustahik (Mampu)    : {dist.get(0, 0)} Observasi ({dist.get(0, 0)/len(df_imputed)*100:.1f}%)")


In [ ]:
# Pembagian Dataset & Pelatihan Model Klasifikasi
features = [
    'pendapatan_per_kapita_bulan', 'skor_aset', 'poverty_percentage',
    'sanitation_access_percent', 'unemployment_rate', 'hdi',
    'population_density', 'radiasi_satelit_ntl'
]
X = df_imputed[features]
y = df_imputed['status_mustahik']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)

# Model Usulan: Random Forest Classifier
model_smart = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=SEED)
model_smart.fit(X_train, y_train)
y_pred_smart = model_smart.predict(X_test)

# Simulasi Pendataan Konvensional (Survei Manual dengan Noise 18%)
y_pred_konv = y_test.copy().values
n_noise = int(0.18 * len(y_pred_konv))
noise_idx = np.random.choice(len(y_pred_konv), size=n_noise, replace=False)
y_pred_konv[noise_idx] = 1 - y_pred_konv[noise_idx]


In [ ]:
# Evaluasi Performa & Kalkulasi Error Zakat
def eval_performance(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred) * 100
    rec = recall_score(y_true, y_pred) * 100
    f1 = f1_score(y_true, y_pred) * 100
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    inc_err = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0.0
    exc_err = (fn / (fn + tp)) * 100 if (fn + tp) > 0 else 0.0
    return {
        'Metode': name,
        'Akurasi (%)': round(acc, 2),
        'Precision (%)': round(prec, 2),
        'Recall (%)': round(rec, 2),
        'F1-Score (%)': round(f1, 2),
        'Inclusion Error (%)': round(inc_err, 2),
        'Exclusion Error (%)': round(exc_err, 2)
    }

res_konv = eval_performance(y_test, y_pred_konv, 'Survei Manual Konvensional')
res_smart = eval_performance(y_test, y_pred_smart, 'SMART-MUSTAHIK (Random Forest)')

df_eval = pd.DataFrame([res_konv, res_smart]).set_index('Metode')
df_eval.to_csv(os.path.join('data', 'evaluasi_performa.csv'))
display(df_eval)


In [ ]:
# Visualisasi Publikasi Grafik 300 DPI
# 1. Plot Utama Perbandingan Performa & Error Zakat
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
colors = ['#d9534f', '#2b8a3e']

df_p1 = df_eval[['Akurasi (%)', 'F1-Score (%)']].reset_index().melt(id_vars='Metode', var_name='Metrik', value_name='Persentase (%)')
sns.barplot(data=df_p1, x='Metrik', y='Persentase (%)', hue='Metode', ax=axes[0], palette=colors)
axes[0].set_title('Perbandingan Akurasi & F1-Score Klasifikasi', fontsize=12, fontweight='bold', pad=10)
axes[0].set_ylim(0, 115)
axes[0].set_xlabel('')
axes[0].set_ylabel('Persentase (%)', fontsize=11)
axes[0].legend(title='', loc='upper left', frameon=True)
for p in axes[0].patches:
    h = p.get_height()
    if h > 0:
        axes[0].annotate(f'{h:.1f}%', (p.get_x() + p.get_width() / 2., h + 1.5), ha='center', va='bottom', fontsize=9.5, fontweight='bold')

df_p2 = df_eval[['Inclusion Error (%)', 'Exclusion Error (%)']].reset_index().melt(id_vars='Metode', var_name='Tipe Error', value_name='Persentase Error (%)')
sns.barplot(data=df_p2, x='Tipe Error', y='Persentase Error (%)', hue='Metode', ax=axes[1], palette=colors)
axes[1].set_title('Perbandingan Inclusion & Exclusion Error Zakat', fontsize=12, fontweight='bold', pad=10)
axes[1].set_ylim(0, 30)
axes[1].set_xlabel('')
axes[1].set_ylabel('Persentase Error (%)', fontsize=11)
axes[1].legend(title='', loc='upper right', frameon=True)
for p in axes[1].patches:
    h = p.get_height()
    if h > 0:
        axes[1].annotate(f'{h:.1f}%', (p.get_x() + p.get_width() / 2., h + 0.6), ha='center', va='bottom', fontsize=9.5, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join('images', 'grafik_hasil_smart_mustahik.png'), dpi=300, bbox_inches='tight')
plt.show()

# 2. Plot Feature Importance
importances = model_smart.feature_importances_
feat_clean = ['Pendapatan per Kapita', 'Skor Aset', 'Kemiskinan (%)', 'Akses Sanitasi (%)', 'Pengangguran (%)', 'IPM', 'Kepadatan Penduduk', 'Radiasi Satelit NTL']
df_imp = pd.DataFrame({'Fitur': feat_clean, 'Importance': importances}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(df_imp['Fitur'], df_imp['Importance'], color='#2b8a3e', edgecolor='#1b5e20', alpha=0.85)
plt.title('Tingkat Kepentingan Fitur (Feature Importance) - Model SMART-MUSTAHIK', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Nilai Importance Relative', fontsize=11)
for i, v in enumerate(df_imp['Importance']):
    plt.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9.5, fontweight='bold')
plt.xlim(0, max(importances) * 1.15)
plt.tight_layout()
plt.savefig(os.path.join('images', 'feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()

# 3. Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred_smart)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Non-Mustahik', 'Mustahik'],
            yticklabels=['Non-Mustahik', 'Mustahik'],
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title('Matriks Konfusi SMART-MUSTAHIK', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Prediksi Model', fontsize=11, fontweight='bold')
plt.ylabel('Ground Truth (Fiqih Kifayah)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('images', 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
